# Data types, units and how data was collected

Core, Part 1 (Statistics I), lesson C1. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** tell categorical from numeric columns, name the unit of observation of a table, tell an observational study from an experiment and name a confounder, and say what in a table could identify a person.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sys
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Unit 1: Kinds of columns

**Predict first.** `order_id` is stored as a whole number. Will `mean()` run on it, and would the result tell you anything? Then: which values does `quantity` take?

In [ ]:
print(orders["order_id"].dtype, orders["order_id"].mean())
print(sorted(order_items["quantity"].unique().tolist()))

## Unit 2: What is one row?

**Predict first.** both lines below are called an average quantity. Will they be equal?

In [ ]:
print(order_items["quantity"].mean().round(2))
print(order_items.groupby("order_id")["quantity"].sum().mean().round(2))

## Unit 3: Watching or assigning

**Predict first.** the table in the question counted every order from January to June. Now count only the orders from April 1 on, when all three stores were open. Will Old Town still be far ahead of Harbor?

In [ ]:
orders["month"] = pd.to_datetime(orders["ordered_at"]).dt.month
named = orders.merge(stores[["store_id", "store_name"]], on="store_id", validate="many_to_one")
print(named[named["month"] >= 4].groupby("store_name").size().to_dict())

## Unit 4: What could identify a person?

**Predict first.** of the 150 customers, how many are the only one with their home city and signup date? And how many of the 1,800 orders belong to those customers?

In [ ]:
same_city_and_day = customers.groupby(["city", "signup_date"])["customer_id"].transform("size")
alone = customers[same_city_and_day == 1]
exposed = orders[orders["customer_id"].isin(alone["customer_id"])]
print(len(alone), len(exposed))

## Finish it

### Exercise 1: Finish it

**Task:** What is the average number of units in a Harbor order? One row of `order_items` is a line, not an order. Use the lines of the orders taken at Harbor, store 3. Put the average in `answer_1`, rounded to 2 decimals.

**Expected:** `answer_1` is a number.

In [ ]:
harbor_orders = orders[orders["store_id"] == 3]["order_id"]  # given: the ids of the Harbor orders
harbor_lines = order_items[order_items["order_id"].isin(harbor_orders)]  # given: their lines
# average the units over orders, not over lines, and set answer_1
answer_1 = None  # replace None

In [ ]:
_ = check("Exercise 1", "answer_1", 2.93, traps={1.61: "That averages over lines. What does one order stand for, and how many units does it hold in all?"})

<details>
<summary>Hint</summary>

Add up the quantities of each order first, then average the sums.

</details>

<details>
<summary>Solution</summary>

```python
harbor_orders = orders[orders["store_id"] == 3]["order_id"]
harbor_lines = order_items[order_items["order_id"].isin(harbor_orders)]
answer_1 = round(float(harbor_lines.groupby("order_id")["quantity"].sum().mean()), 2)
```

2.93 units in a Harbor order, averaged over its 355 orders. The average over its 645 lines is 1.61 units for each line, which the check cell reports: the unit of observation changed from order to line.

</details>

## On your own

### Exercise 2: On your own

**Task:** Which of these four studies are experiments, and which are observational? Before you start, say in one sentence what the result looks like: four words. (a) A coin flip decides, for each day, whether the till suggests a pastry; the owner compares pastry sales on the two kinds of day. (b) The owner compares the pastry sales of orders paid by mobile with those paid in cash. (c) The owner reads last year's records to see whether gold customers visit more often than bronze ones. (d) Half of the loyalty customers, chosen by drawing lots, get a free-cookie coupon, and the owner counts their visits next month against the others. Put one word for each study in `answer_2`, in the order a, b, c, d: `"experiment"` or `"observational"`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of four words.

In [ ]:
_ = check("Exercise 2", "answer_2", ["experiment", "observational", "observational", "experiment"], traps={("observational", "observational", "observational", "observational"): "Four studies cannot all be observational: in two of them the owner decides who gets the treatment. Which two?"})

<details>
<summary>Hint</summary>

Ask of each study: did the owner assign the groups, or only watch which group each case fell into?

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = ["experiment", "observational", "observational", "experiment"]
```

(a) and (d) are experiments, and randomized ones: the owner assigns the treatment, by a coin flip and by drawing lots. (b) and (c) are observational: the groups, payment method and tier, were there before the owner looked. All five tables of cafe-sales are observational data of the (b) and (c) kind.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wants to report the typical loyalty tier of the 150 customers, numbers the tiers 1, 2 and 3, and writes that the average customer sits between bronze and silver. Report the tiers the way their kind allows: set `answer_3` to the share of customers in each tier, in the order bronze, silver, gold, each rounded to 2 decimals.

**Expected:** `answer_3` is a list of three shares.

In [ ]:
tier_number = customers["loyalty_tier"].map({"bronze": 1, "silver": 2, "gold": 3})  # the colleague's numbers
answer_3 = round(float(tier_number.mean()), 2)  # the colleague's average tier

In [ ]:
_ = check("Exercise 3", "answer_3", [0.57, 0.33, 0.1], traps={1.53: "An average of tier numbers treats the steps between tiers as equal. What can you report for a column of ordered labels?"})

<details>
<summary>Hint</summary>

A column of labels, even ordered ones, is summarized by counting: how many in each level, as a share of all.

</details>

<details>
<summary>Solution</summary>

```python
shares = customers["loyalty_tier"].value_counts(normalize=True)
answer_3 = shares.reindex(["bronze", "silver", "gold"]).round(2).tolist()
```

57 percent bronze, 33 percent silver and 10 percent gold. The numbers 1, 2 and 3 were a label code, and nothing says gold is three times bronze; the average 1.53 came out of that code, not out of the customers. The check cell reports the colleague's 1.53.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")